# Identity and agent for per-user news access

Create the Entra objects and the agent that the rest of lab 08-05c runs on, then check that each user can call the agent with their own token.

[08-05c-00](08-05c-00-news-mcp-per-user-access.md) explains the design. One agent, `news-desk-agent`, serves two users who may read different news categories. A user's categories are Entra app roles on a resource app, `news-mcp-{suffix}`. When the agent calls a news tool, Foundry exchanges the user's token for one addressed to `news-mcp-{suffix}`, on behalf of that user (`AgentOnBehalfOfUser`), and the server in front of GNews reads the roles from that token. [08-05c-02](08-05c-02-wrapper-container-app.ipynb) and [08-05c-03](08-05c-03-apim-passthrough.ipynb) build the two servers.

| | User A | User B |
|---|---|---|
| Who | you, the admin running this notebook | `news-sports-desk`, created in Step 5 |
| App roles on `news-mcp-{suffix}` | `News.Read.All` | `News.Read.Sports`, `News.Read.Science`, `News.Read.Health` |
| Signs in with | your own account | a Temporary Access Pass |

**What this notebook does:**
1. Registers the resource app `news-mcp-{suffix}`: the `News.Read` scope and ten app roles
2. Registers the public client app `news-desk-client-{suffix}` that both users sign in to
3. Signs you in through that client app for the two Graph permissions the Azure CLI token lacks
4. Turns on the Temporary Access Pass method if it is off
5. Creates user B and a pass for them
6. Signs user B in once
7. Creates `news-desk-agent` and lets its endpoint accept user tokens through the agent blueprint
8. Consents the blueprint to `News.Read` on the resource app
9. Assigns the app roles
10. Creates the `news-mcp-apim` connection with delegated authentication
11. Calls the agent as user A and as user B
12. Lists what this notebook wrote to `.env`

## Prerequisites

1. **Python environment**: run `uv sync` from the repository root and select the `.venv` kernel.
2. **Core gateway**: [05-02](../../05-foundry-project-pattern-setup/05-02-deploy-foundry-core-gateway/05-02-01-deploy-foundry-core-gateway.ipynb) deployed `aif-core-{suffix}` with the admin project `project-admin-{suffix}`, and wrote `CHAT_MODEL` to `.env`.
3. **Azure CLI**: `az login` as a **Global Administrator** of a tenant you are allowed to change. The notebook registers apps, creates a user, grants tenant-wide consent, assigns app roles and may turn on the Temporary Access Pass method. Teardown at the end removes all of it except that tenant setting.
4. **Entra licences**: none. App roles are assigned to users directly; assigning them to groups would need Entra ID P1.
5. **A private browser window** for user B's sign-in in Step 6, so it does not mix with your admin session.

Both users sign in with the device code flow: the cell prints a code, you enter it at `https://microsoft.com/devicelogin`. It works when the kernel runs on another machine (Codespaces, SSH, a dev box), where a browser redirect to `localhost` would not reach it. On your own machine, `checks.user_credential(..., device_code=False)` opens a browser instead.

## Step 0: Configuration

In [1]:
import base64, hashlib, json, os, secrets, subprocess, time
from pathlib import Path

import requests
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

import news_mcp_checks as checks
import news_mcp_identity as ident
from news_mcp_identity import authz

repo_root = Path(subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip())
ENV_FILE = repo_root / '.env'
load_dotenv(ENV_FILE, override=True)

account = json.loads(subprocess.run('az account show -o json', shell=True, capture_output=True, text=True).stdout)
SUBSCRIPTION_ID = account['id']
TENANT_ID = account['tenantId']
# Suffix: first 6 hex chars of SHA-256 of the subscription id, as in the core gateway lab
SUFFIX = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
CHAT_MODEL = os.environ.get('CHAT_MODEL', 'gpt-5.4-mini')
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'
AGENT_NAME = 'news-desk-agent'

cli = AzureCliCredential()
ARM, FOUNDRY = 'https://management.azure.com/.default', 'https://ai.azure.com/.default'


def graph(method, path, body=None, token=None):
    """Microsoft Graph v1.0, as the Azure CLI user unless another token is given."""
    token = token or cli.get_token('https://graph.microsoft.com/.default').token
    r = requests.request(method, f'https://graph.microsoft.com/v1.0{path}', json=body,
                         headers={'Authorization': f'Bearer {token}'}, timeout=60)
    if not r.ok:
        raise RuntimeError(f'{method} {path}: HTTP {r.status_code} {r.text[:600]}')
    return r.json() if r.content else None


def call(method, url, scope, body=None, content_type='application/json'):
    """ARM or the Foundry data plane with a CLI token for `scope`. The caller checks the response."""
    return requests.request(method, url, data=None if body is None else json.dumps(body), timeout=120,
                            headers={'Authorization': f'Bearer {cli.get_token(scope).token}', 'Content-Type': content_type})


def find_one(collection, odata_filter):
    found = graph('GET', f'/{collection}?$filter={odata_filter}')['value']
    return found[0] if found else None


def env_set(**values):
    """Write keys to the repo .env: replace the key's line if present, append it otherwise."""
    lines = ENV_FILE.read_text().splitlines() if ENV_FILE.exists() else []
    for key, value in values.items():
        os.environ[key] = value
        at = [i for i, line in enumerate(lines) if line.split('=', 1)[0].strip() == key]
        if at:
            lines[at[0]] = f'{key}={value}'
        else:
            lines.append(f'{key}={value}')
    ENV_FILE.write_text('\n'.join(lines) + '\n')


def jwt_claims(token):
    """Decode a JWT payload for display. No signature check: the token came straight from Entra."""
    payload = token.split('.')[1]
    return json.loads(base64.urlsafe_b64decode(payload + '=' * (-len(payload) % 4)))


ADMIN = graph('GET', '/me?$select=id,userPrincipalName')
domains = graph('GET', '/organization?$select=verifiedDomains')['value'][0]['verifiedDomains']
NEWS_TEST_USER_UPN = os.environ.get('NEWS_TEST_USER_UPN') or f"news-sports-desk@{next(d['name'] for d in domains if d['isDefault'])}"

print(f'Suffix        : {SUFFIX}')
print(f'Admin project : {PROJECT_ENDPOINT}')
print(f'Model         : {CHAT_MODEL}')
print(f"User A        : {ADMIN['userPrincipalName']}")
print(f'User B        : {NEWS_TEST_USER_UPN}')
print(f'Resource app  : news-mcp-{SUFFIX} ({authz.app_id_uri(SUFFIX)})')
print(f'Client app    : news-desk-client-{SUFFIX}')
print(f'Agent         : {AGENT_NAME}')

Suffix        : c2676f
Admin project : https://aif-core-c2676f.services.ai.azure.com/api/projects/project-admin-c2676f
Model         : gpt-5.4-mini
User A        : {user}@{tenant}.onmicrosoft.com
User B        : {user}@{tenant}.onmicrosoft.com
Resource app  : news-mcp-c2676f (api://news-mcp-c2676f)
Client app    : news-desk-client-c2676f
Agent         : news-desk-agent


## Step 1: The resource app `news-mcp-{suffix}`

The resource app stands for the news server. Its manifest comes from `news_mcp_identity.resource_app_manifest`:

- **Identifier URI** `api://news-mcp-{suffix}` and **`requestedAccessTokenVersion: 2`**, so tokens for it are v2 and carry the app's client id as `aud`. Many tenants refuse App ID URIs without a tenant identifier, but exempt apps that ask for v2 tokens; sending both in one request keeps the readable URI.
- **One delegated scope, `News.Read`.** A token for the scope proves the caller may use the API at all. Which categories they get is the app roles' job.
- **Ten app roles**, `News.Read.All` and `News.Read.<Category>` for the nine GNews categories. `allowedMemberTypes` is `User` and `Application`: users get them here, and 08-05c-02 also assigns them to agent identities for the per-agent variant.

Role and scope ids are fixed GUIDs derived from the URI (`authz.permission_id`). Entra refuses a PATCH that would remove an enabled role, so a re-run must send the same ids.

In [4]:
def ensure_app(manifest):
    """Create the app registration and its service principal if absent; otherwise bring the app to the manifest."""
    app = find_one('applications', f"displayName eq '{manifest['displayName']}'")
    if app is None:
        app = graph('POST', '/applications', manifest)
        print(f"  created {manifest['displayName']}")
    else:
        graph('PATCH', f"/applications/{app['id']}", manifest)
        print(f"  updated {manifest['displayName']}")
    sp = find_one('servicePrincipals', f"appId eq '{app['appId']}'") or graph('POST', '/servicePrincipals', {'appId': app['appId']})
    return app, sp


resource_app, resource_sp = ensure_app(ident.resource_app_manifest(SUFFIX))
RESOURCE_APP_ID, RESOURCE_SP_ID = resource_app['appId'], resource_sp['id']
env_set(NEWS_MCP_RESOURCE_APP_ID=RESOURCE_APP_ID, NEWS_MCP_RESOURCE_SP_ID=RESOURCE_SP_ID)

app = graph('GET', f"/applications/{resource_app['id']}?$select=identifierUris,api,appRoles")
print(f"Identifier URI : {app['identifierUris'][0]}")
print(f"Token version  : v{app['api']['requestedAccessTokenVersion']}")
print(f"Scope          : {', '.join(s['value'] for s in app['api']['oauth2PermissionScopes'])}")
print(f"App roles      : {', '.join(r for r in authz.ROLES if r in {a['value'] for a in app['appRoles']})}")

  updated news-mcp-c2676f
Identifier URI : api://news-mcp-c2676f
Token version  : v2
Scope          : News.Read
App roles      : News.Read.All, News.Read.General, News.Read.World, News.Read.Nation, News.Read.Business, News.Read.Technology, News.Read.Entertainment, News.Read.Sports, News.Read.Science, News.Read.Health


## Step 2: The client app `news-desk-client-{suffix}`

Users sign in to this public client (no secret, redirect `http://localhost`, device code allowed through `isFallbackPublicClient`). It asks for three things:

| Resource | Delegated permission | Used for |
|---|---|---|
| `news-mcp-{suffix}` | `News.Read` | the direct `tools/list` check in 08-05c-02 |
| Microsoft Graph | `User.Read` | user B's test sign-in in Step 6 |
| Microsoft Graph | `Policy.ReadWrite.AuthenticationMethod`, `UserAuthenticationMethod.ReadWrite.All` | your Temporary Access Pass calls in Steps 4 and 5 |
| the agent blueprint | `access_agent` | calling the agent (added in Step 7, once the blueprint exists) |

Delegated permissions act with the signed-in user's own rights and never more. User B holds no directory role, so the two Graph admin permissions give them nothing.

Admin consent is an `oauth2PermissionGrant` with `consentType: AllPrincipals` for each resource, so no user sees a consent prompt.

In [3]:
def grant(client_sp_id, resource_sp_id, scopes):
    """Tenant-wide admin consent: create the grant, or set its scopes if it already exists."""
    body = ident.permission_grant_body(client_sp_id, resource_sp_id, scopes)
    existing = find_one('oauth2PermissionGrants', f"clientId eq '{client_sp_id}' and resourceId eq '{resource_sp_id}' "
                                                  "and consentType eq 'AllPrincipals'")
    if existing:
        graph('PATCH', f"/oauth2PermissionGrants/{existing['id']}", {'scope': body['scope']})
    else:
        graph('POST', '/oauth2PermissionGrants', body)


def show_grants(client_sp_id):
    for g in graph('GET', f"/oauth2PermissionGrants?$filter=clientId eq '{client_sp_id}'")['value']:
        resource = graph('GET', f"/servicePrincipals/{g['resourceId']}?$select=displayName")['displayName']
        print(f"  {resource:<40} {g['consentType']:<14} {g['scope'].strip()}")


client_app, client_sp = ensure_app(ident.client_app_manifest(SUFFIX, RESOURCE_APP_ID))
CLIENT_APP_ID, CLIENT_SP_ID = client_app['appId'], client_sp['id']
env_set(NEWS_DESK_CLIENT_APP_ID=CLIENT_APP_ID)

GRAPH_SP_ID = find_one('servicePrincipals', f"appId eq '{ident.GRAPH_APP_ID}'")['id']
grant(CLIENT_SP_ID, GRAPH_SP_ID, list(ident.GRAPH_SCOPE_IDS))
grant(CLIENT_SP_ID, RESOURCE_SP_ID, [authz.SCOPE])
print(f'Consent granted to news-desk-client-{SUFFIX}:')
show_grants(CLIENT_SP_ID)

  created news-desk-client-c2676f
Consent granted to news-desk-client-c2676f:
  Microsoft Graph                          AllPrincipals  User.Read Policy.ReadWrite.AuthenticationMethod UserAuthenticationMethod.ReadWrite.All
  news-mcp-c2676f                          AllPrincipals  News.Read


## Step 3: A Graph token for the Temporary Access Pass calls

Reading or changing the Temporary Access Pass policy needs `Policy.ReadWrite.AuthenticationMethod`, and issuing a pass needs `UserAuthenticationMethod.ReadWrite.All`. The Azure CLI token for Graph does not carry either: the first cell asks the CLI for the first one and prints what it answers.

The second cell signs you in through `news-desk-client` instead, which Step 2 consented for both. Run it, open `https://microsoft.com/devicelogin`, enter the code it prints and sign in with your admin account.

In [5]:
probe = subprocess.run(
    'az account get-access-token --scope https://graph.microsoft.com/Policy.ReadWrite.AuthenticationMethod -o json',
    shell=True, capture_output=True, text=True)
if probe.returncode == 0:
    scp = jwt_claims(json.loads(probe.stdout)['accessToken']).get('scp', '')
    print(f"Azure CLI token issued; carries Policy.ReadWrite.AuthenticationMethod: {'Policy.ReadWrite.AuthenticationMethod' in scp.split()}")
else:
    print('Azure CLI token refused:')
    print('  ' + next(line for line in probe.stderr.splitlines() if 'AADSTS' in line or 'ERROR' in line)[:400])

Azure CLI token refused:
  ERROR: AADSTS65002: Consent between first party application '00000000-0000-0000-0000-000000000000' and first party resource '00000000-0000-0000-0000-000000000000' must be configured via preauthorization - applications owned and operated by Microsoft must get approval from the API owner before requesting tokens for that API. Trace ID: 00000000-0000-0000-0000-000000000000 Correlation ID: 3e3d939e-c6


**Observed on 2026-10-30.** The Azure CLI refused with `AADSTS65002`: consent between Microsoft's own CLI app and Graph "must be configured via preauthorization", which no tenant admin can add. A delegated permission on the lab's own client app is the way around it.

In [6]:
TAP_SCOPES = ['https://graph.microsoft.com/Policy.ReadWrite.AuthenticationMethod',
              'https://graph.microsoft.com/UserAuthenticationMethod.ReadWrite.All']
admin_cred = checks.user_credential(TENANT_ID, CLIENT_APP_ID, device_code=True)


def admin_graph():
    """Your Graph token through news-desk-client. Refreshed silently after the first sign-in."""
    return admin_cred.get_token(*TAP_SCOPES).token


claims = jwt_claims(admin_graph())
print(f"\nSigned in as {claims['upn']}")
print(f"Delegated permissions in the token: {claims['scp']}")

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code A2RAXX5DY to authenticate.

Signed in as {user}@{tenant}.onmicrosoft.com
Delegated permissions in the token: Policy.ReadWrite.AuthenticationMethod User.Read UserAuthenticationMethod.ReadWrite.All profile openid email


## Step 4: Turn on the Temporary Access Pass method

User B must pass the tenant's multifactor sign-in policy without a phone. A Temporary Access Pass is a time-limited passcode an admin issues, and it counts as multifactor authentication for Conditional Access. The method is off by default.

If it is off, the cell turns it on for all users, allows multi-use passes and caps their lifetime at 8 hours. If it is already on with settings that cannot issue a multi-use 8-hour pass to a new user, the cell stops and leaves your settings alone. Teardown does not switch the method off again; the Cleanup section has the line if you want to.

In [7]:
TAP_POLICY = '/policies/authenticationMethodsPolicy/authenticationMethodConfigurations/temporaryAccessPass'


def tap_summary(policy):
    return (f"state={policy['state']}  usable once={policy['isUsableOnce']}  "
            f"lifetime {policy['minimumLifetimeInMinutes']} to {policy['maximumLifetimeInMinutes']} min  "
            f"targets={[t['id'] for t in policy.get('includeTargets', [])]}")


policy = graph('GET', TAP_POLICY, token=admin_graph())
print(f'Before: {tap_summary(policy)}')
if policy['state'] != 'enabled':
    graph('PATCH', TAP_POLICY, ident.tap_policy_patch(), token=admin_graph())
    policy = graph('GET', TAP_POLICY, token=admin_graph())
    print(f'After : {tap_summary(policy)}')
elif policy['isUsableOnce'] or policy['maximumLifetimeInMinutes'] < ident.tap_method_body()['lifetimeInMinutes'] \
        or 'all_users' not in [t['id'] for t in policy.get('includeTargets', [])]:
    raise RuntimeError('The pass method is on, but cannot issue a multi-use 8-hour pass to every user. '
                       'Change it in the Entra admin center or use the per-agent variant in 08-05c-02.')

Before: state=enabled  usable once=False  lifetime 60 to 480 min  targets=['all_users']


**Observed on 2026-10-30.** The method was already on in the tenant this lab was built in (multi-use allowed, 60 to 480 minutes, all users), so the cell changed nothing and teardown leaves it as it was.

## Step 5: Create user B and a pass

`news-sports-desk` gets a random password that nobody ever uses, then one multi-use pass valid for 8 hours. Tokens from a pass sign-in end when the pass does, so a new run of this cell (which replaces the pass) is how you extend a session. The pass goes to `.env` as `NEWS_TEST_USER_TAP` and is never printed.

In [11]:
user = find_one('users', f"userPrincipalName eq '{NEWS_TEST_USER_UPN}'")
if user is None:
    user = graph('POST', '/users', {
        'accountEnabled': True,
        'displayName': 'News Sports Desk',
        'mailNickname': 'news-sports-desk',
        'userPrincipalName': NEWS_TEST_USER_UPN,
        'passwordProfile': {'password': secrets.token_urlsafe(24) + 'aA1!', 'forceChangePasswordNextSignIn': False},
    })
    print(f'Created {NEWS_TEST_USER_UPN}')
TEST_USER_ID = user['id']
env_set(NEWS_TEST_USER_UPN=NEWS_TEST_USER_UPN, NEWS_TEST_USER_ID=TEST_USER_ID)

# A new user reaches the authentication methods service a little later; until then it answers 403.
methods = f'/users/{TEST_USER_ID}/authentication/temporaryAccessPassMethods'
for attempt in range(12):
    try:
        existing = graph('GET', methods, token=admin_graph())['value']
        break
    except RuntimeError as e:
        if 'HTTP 403' not in str(e) or attempt == 11:
            raise
        time.sleep(10)

# A user holds at most one pass: replace any earlier one.
for old in existing:
    graph('DELETE', f"{methods}/{old['id']}", token=admin_graph())
tap = graph('POST', methods, ident.tap_method_body(), token=admin_graph())
env_set(NEWS_TEST_USER_TAP=tap['temporaryAccessPass'])
print(f"Pass written to .env as NEWS_TEST_USER_TAP: multi-use={not tap['isUsableOnce']}, "
      f"{tap['lifetimeInMinutes']} minutes from {tap['startDateTime']}")

Pass written to .env as NEWS_TEST_USER_TAP: multi-use=True, 480 minutes from 2026-10-30T16:32:14.2900446Z


**Observed on 2026-10-30.** The first pass request for the new user returned HTTP 403 `accessDenied`, although the token carried `UserAuthenticationMethod.ReadWrite.All` and the caller was a Global Administrator. The same request succeeded about 15 seconds later: the user had not reached the authentication methods service yet. The loop above waits for that.

## Step 6: Sign in as user B

Open a private browser window, go to `https://microsoft.com/devicelogin` and enter the code the cell prints. Sign in as user B with the pass from `.env` (`grep NEWS_TEST_USER_TAP .env`). The token is for Graph `User.Read`, which Step 2 consented; the cell prints who signed in and how (`amr`).

In [12]:
user_b_cred = checks.user_credential(TENANT_ID, CLIENT_APP_ID, device_code=True)
claims = jwt_claims(user_b_cred.get_token('https://graph.microsoft.com/User.Read').token)
print(f"\nSigned in as {claims['upn']}")
print(f"Authentication methods (amr): {claims['amr']}")

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code EFS8WHM4B to authenticate.

Signed in as {user}@{tenant}.onmicrosoft.com
Authentication methods (amr): ['pwd', 'mfa']


**Observed on 2026-10-30.** User B signed in with the pass in a private window. The token's `amr` was `pwd, mfa`: the pass counted as multifactor authentication for the tenant's Conditional Access policy. No security-info registration prompt appeared, although the tenant's Microsoft-managed policies include one for security info registration.

## Step 7: The agent and its endpoint

`news-desk-agent` is a prompt agent with no tools yet; 08-05c-02 and 08-05c-03 add them. Its instructions tell the model to report a denial word for word and not to try another category.

Foundry gives every agent an **agent identity** (`instance_identity`) and an **agent blueprint** (`blueprint`), both in Entra. The endpoint accepts Foundry tokens by default (the `AzureRbac` scheme, still shown as its alias `Entra` on agents created before the rename). The PATCH adds the `AgentBlueprint` scheme, which accepts a user's token for `api://<blueprint-client-id>/access_agent` from the clients listed in `allowed_clients`. A PATCH replaces the whole list, so it repeats `AzureRbac`.

The client app then gets `access_agent` on the blueprint, with admin consent.

In [21]:
# allow_preview: get_openai_client(agent_name=...) in Step 11 is a preview operation.
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=cli, allow_preview=True)
INSTRUCTIONS = (
    'You are a news desk assistant. Answer requests for headlines with the news tools and keep answers short. '
    'If a news tool call is denied, tell the user exactly what the tool returned and do not try another '
    'category or tool for that request.'
)
agent = project.agents.create_version(
    agent_name=AGENT_NAME, definition=PromptAgentDefinition(model=CHAT_MODEL, instructions=INSTRUCTIONS))


def agent_get(name):
    r = call('GET', f'{PROJECT_ENDPOINT}/agents/{name}?api-version={ident.AGENT_API_VERSION}', FOUNDRY)
    r.raise_for_status()
    return r.json()


latest = agent_get(AGENT_NAME)['versions']['latest']
BLUEPRINT_CLIENT_ID = latest['blueprint']['client_id']
BLUEPRINT_NAME = latest['blueprint_reference']['blueprint_id']
env_set(NEWS_AGENT_BLUEPRINT_CLIENT_ID=BLUEPRINT_CLIENT_ID)
print(f'{AGENT_NAME} v{agent.version}')
print(f"  agent identity : principal {latest['instance_identity']['principal_id']}, client {latest['instance_identity']['client_id']}")
print(f'  blueprint      : {BLUEPRINT_NAME}, client {BLUEPRINT_CLIENT_ID}')

r = call('PATCH', f'{PROJECT_ENDPOINT}/agents/{AGENT_NAME}?api-version={ident.AGENT_API_VERSION}', FOUNDRY,
         ident.endpoint_schemes_body(CLIENT_APP_ID), ident.ENDPOINT_PATCH_CONTENT_TYPE)
print(f'\nPATCH endpoint authorization schemes: HTTP {r.status_code}')
r.raise_for_status()
for scheme in agent_get(AGENT_NAME)['agent_endpoint']['authorization_schemes']:
    print(f"  {scheme['type']:<15} {json.dumps(scheme.get('token_validation_rules', {}))}")

news-desk-agent v1
  agent identity : principal 00000000-0000-0000-0000-000000000000, client 00000000-0000-0000-0000-000000000000
  blueprint      : news-desk-agent-0d9d4, client 00000000-0000-0000-0000-000000000000

PATCH endpoint authorization schemes: HTTP 200
  AzureRbac       {}
  AgentBlueprint  {"allowed_clients": ["00000000-0000-0000-0000-000000000000"]}


In [14]:
blueprint_app = find_one('applications', f"appId eq '{BLUEPRINT_CLIENT_ID}'")
blueprint_scopes = {s['value']: s['id'] for s in blueprint_app['api']['oauth2PermissionScopes']}
print(f"Blueprint app: {blueprint_app['@odata.type']}")
print(f"  identifier URIs : {blueprint_app['identifierUris']}")
print(f"  scopes          : {list(blueprint_scopes)}")

BLUEPRINT_SP_ID = find_one('servicePrincipals', f"appId eq '{BLUEPRINT_CLIENT_ID}'")['id']
ensure_app(ident.client_app_manifest(SUFFIX, RESOURCE_APP_ID, BLUEPRINT_CLIENT_ID, blueprint_scopes['access_agent']))
grant(CLIENT_SP_ID, BLUEPRINT_SP_ID, ['access_agent'])
print(f'\nConsent granted to news-desk-client-{SUFFIX}:')
show_grants(CLIENT_SP_ID)

Blueprint app: #microsoft.graph.agentIdentityBlueprint
  identifier URIs : []
  scopes          : ['access_agent']
  updated news-desk-client-c2676f

Consent granted to news-desk-client-c2676f:
  Microsoft Graph                          AllPrincipals  User.Read Policy.ReadWrite.AuthenticationMethod UserAuthenticationMethod.ReadWrite.All
  news-mcp-c2676f                          AllPrincipals  News.Read
  aif-core-c2676f-project-admin-c2676f-news-desk-agent-0d9d4-AgentIdentityBlueprint AllPrincipals  access_agent


**Observed on 2026-10-30.** After the PATCH the blueprint carried the `access_agent` scope; the blueprint of an older agent in the same project, whose endpoint still had only the `Entra` scheme, carried none. The blueprint's `identifierUris` stayed empty, and Entra still issued tokens for `api://<blueprint-client-id>/access_agent` in Step 11.

## Step 8: Consent the blueprint to `News.Read`

When the agent calls a tool for user B, the agent identity asks Entra for a `News.Read` token on user B's behalf. Without consent, Entra would answer with a consent request the user would have to follow. Three settings remove that step:

1. **Preauthorization** on the resource app: `api.preAuthorizedApplications` names the blueprint and the `News.Read` scope id.
2. **The blueprint's own permissions**, set through Foundry: `requiredResourceAccess` declares `News.Read`, and `inheritablePermissions` lets scopes consented on the blueprint flow to its agent identities.
3. **Admin consent on the blueprint principal**: an `AllPrincipals` grant for `News.Read`. Interactive consent on a child agent identity fails with `AADSTS82014`, so consent goes on the blueprint.

In [15]:
graph('PATCH', f"/applications/{resource_app['id']}", ident.preauthorization_patch(SUFFIX, BLUEPRINT_CLIENT_ID))
preauth = graph('GET', f"/applications/{resource_app['id']}?$select=api")['api']['preAuthorizedApplications']
print(f"1. preAuthorizedApplications: {[(p['appId'], p['delegatedPermissionIds']) for p in preauth]}")

r = call('PATCH', f'{PROJECT_ENDPOINT}/managedAgentIdentityBlueprints/{BLUEPRINT_NAME}?api-version={ident.BLUEPRINT_API_VERSION}',
         FOUNDRY, ident.blueprint_permissions_body(SUFFIX, RESOURCE_APP_ID))
print(f'2. blueprint permissions PATCH: HTTP {r.status_code}')
r.raise_for_status()

grant(BLUEPRINT_SP_ID, RESOURCE_SP_ID, [authz.SCOPE])
print('3. admin consent on the blueprint principal:')
show_grants(BLUEPRINT_SP_ID)

1. preAuthorizedApplications: [('00000000-0000-0000-0000-000000000000', ['00000000-0000-0000-0000-000000000000'])]
2. blueprint permissions PATCH: HTTP 200
3. admin consent on the blueprint principal:
  news-mcp-c2676f                          AllPrincipals  News.Read


## Step 9: Assign the app roles

Each assignment is a POST to `appRoleAssignedTo` on the resource app's service principal. The roles appear in the `roles` claim of the next token either user gets for `news-mcp-{suffix}`; a token already issued keeps the roles it had.

In [17]:
WANTED = {ADMIN['id']: ['News.Read.All'], TEST_USER_ID: ['News.Read.Sports', 'News.Read.Science', 'News.Read.Health']}
LABELS = {ADMIN['id']: 'user A (you)', TEST_USER_ID: 'user B (news-sports-desk)'}
ROLE_BY_ID = {authz.permission_id(SUFFIX, role): role for role in authz.ROLES}

assigned_to = f'/servicePrincipals/{RESOURCE_SP_ID}/appRoleAssignedTo'
have = {(a['principalId'], a['appRoleId']) for a in graph('GET', assigned_to)['value']}
for principal, roles in WANTED.items():
    for role in roles:
        body = ident.role_assignment_body(SUFFIX, principal, RESOURCE_SP_ID, role)
        if (principal, body['appRoleId']) not in have:
            graph('POST', assigned_to, body)

print(f'App role assignments on news-mcp-{SUFFIX}:')
for a in graph('GET', assigned_to)['value']:
    print(f"  {LABELS.get(a['principalId'], a['principalDisplayName']):<28} {ROLE_BY_ID[a['appRoleId']]}")

App role assignments on news-mcp-c2676f:
  user A (you)                 News.Read.All
  user B (news-sports-desk)    News.Read.Sports
  user B (news-sports-desk)    News.Read.Science
  user B (news-sports-desk)    News.Read.Health


## Step 10: The `news-mcp-apim` connection

A connection tells Foundry where an MCP server is and how to authenticate to it. Learn's user delegation page gives the connection JSON (`authType: AgentOnBehalfOfUser` plus `scopes`) but not the API that accepts it, and the published ARM schema has neither the value nor the `scopes` field. The cell tries the ARM PUT with that JSON at three api-versions, then with `AgentUserImpersonation` (the closest value the schema does list), and keeps the first one ARM accepts. `.env` records which, for 08-05c-02.

The target is the MCP API that 08-05c-03 adds to the core gateway. It does not exist yet; a connection is only a record.

In [18]:
APIM_MCP_URL = f'https://apim-foundry-{SUFFIX}.azure-api.net/news-mcp'

for auth_type, version in ident.connection_attempts():
    r = call('PUT', ident.connection_url(SUBSCRIPTION_ID, SUFFIX, 'news-mcp-apim', version), ARM,
             ident.delegated_connection_body(APIM_MCP_URL, SUFFIX, auth_type))
    detail = '' if r.ok else (r.json().get('error', {}).get('message') or r.text)[:240]
    print(f'  {auth_type:<24} {version:<20} HTTP {r.status_code}  {detail}')
    if r.ok:
        break
else:
    raise RuntimeError('ARM accepted no delegated connection shape; see "If the connection API refuses" in 08-05c-00.')

env_set(NEWS_CONNECTION_AUTH_TYPE=auth_type, NEWS_CONNECTION_API_VERSION=version)
stored = call('GET', ident.connection_url(SUBSCRIPTION_ID, SUFFIX, 'news-mcp-apim', version), ARM).json()['properties']
print(f"\nStored: authType={stored.get('authType')}  scopes={stored.get('scopes')}  target={stored.get('target')}")

  AgentOnBehalfOfUser      2025-04-01-preview   HTTP 200  

Stored: authType=AgentOnBehalfOfUser  scopes=['api://news-mcp-c2676f/News.Read']  target=https://apim-foundry-c2676f.azure-api.net/news-mcp


**Observed on 2026-10-30.** ARM accepted the Learn JSON on the first attempt, at `2025-04-01-preview`, and stored both `authType: AgentOnBehalfOfUser` and `scopes`, although the published ARM schema lists neither. The other five attempts did not run. Whether Foundry uses the connection as intended shows in 08-05c-02, when the first tool call reaches the wrapper.

## Step 11: Call the agent as each user

`checks.agent_client` builds a plain `OpenAI` client whose base URL is the agent's endpoint (`.../agents/news-desk-agent/endpoint/protocols/openai`) and whose bearer token is the user's token for `api://<blueprint-client-id>/access_agent`. Both users signed in already, so the tokens come without a prompt.

The last call uses a Foundry token through `project.get_openai_client`, which the `AzureRbac` scheme still accepts.

In [22]:
PROMPT = 'Who are you and what can you do? Two sentences.'
for label, cred in (('User A', admin_cred), ('User B', user_b_cred)):
    client = checks.agent_client(cred, PROJECT_ENDPOINT, AGENT_NAME, BLUEPRINT_CLIENT_ID)
    print(f'{label} (AgentBlueprint scheme): {client.responses.create(input=PROMPT).output_text}\n')

foundry_client = project.get_openai_client(agent_name=AGENT_NAME)
print(f'Foundry token (AzureRbac scheme): {foundry_client.responses.create(input=PROMPT).output_text}')

User A (AgentBlueprint scheme): I’m a news desk assistant that can help you get current headlines and news-related information using news tools. I keep responses short and, when a news tool is available and allowed, I can use it to fetch relevant headlines.

User B (AgentBlueprint scheme): I’m a news desk assistant that helps answer requests using news tools. I can find and summarize headlines or current news-related information, but I keep responses short.

Foundry token (AzureRbac scheme): I’m a news desk assistant that can fetch current headlines and help answer news-related requests. I can give you short, up-to-date summaries from the news tools when available.


**Observed on 2026-10-30.** Learn's Python example builds the client from `base_url` and the token provider alone; the endpoint then answers `400 Missing required query parameter: api-version`. `checks.agent_client` sends `api-version=v1`, as `get_openai_client` does. `get_openai_client(agent_name=...)` also needs `allow_preview=True` on the project client.

## Step 12: What 08-05c-02 and 08-05c-03 read from `.env`

In [23]:
WRITTEN = ['NEWS_MCP_RESOURCE_APP_ID', 'NEWS_MCP_RESOURCE_SP_ID', 'NEWS_DESK_CLIENT_APP_ID', 'NEWS_TEST_USER_UPN',
           'NEWS_TEST_USER_ID', 'NEWS_TEST_USER_TAP', 'NEWS_AGENT_BLUEPRINT_CLIENT_ID',
           'NEWS_CONNECTION_AUTH_TYPE', 'NEWS_CONNECTION_API_VERSION']
for key in WRITTEN:
    print(f"  {key:<32} {'(hidden)' if key == 'NEWS_TEST_USER_TAP' else os.environ[key]}")

  NEWS_MCP_RESOURCE_APP_ID         00000000-0000-0000-0000-000000000000
  NEWS_MCP_RESOURCE_SP_ID          00000000-0000-0000-0000-000000000000
  NEWS_DESK_CLIENT_APP_ID          00000000-0000-0000-0000-000000000000
  NEWS_TEST_USER_UPN               {user}@{tenant}.onmicrosoft.com
  NEWS_TEST_USER_ID                00000000-0000-0000-0000-000000000000
  NEWS_TEST_USER_TAP               (hidden)
  NEWS_AGENT_BLUEPRINT_CLIENT_ID   00000000-0000-0000-0000-000000000000
  NEWS_CONNECTION_AUTH_TYPE        AgentOnBehalfOfUser
  NEWS_CONNECTION_API_VERSION      2025-04-01-preview


## Cleanup

Run this after 08-05c-02 and 08-05c-03; both point here. Uncomment the cell to delete, in order, everything lab 08-05c created: the three agents, the four connections, the MCP API and named value on the core gateway, the Track A resource group, user B, and both app registrations (their service principals, consent grants and role assignments go with them). Users and apps are then purged from the Entra recycle bin, so a later run can reuse their names. The last lines remove the keys this lab wrote to `.env`.

The Temporary Access Pass method stays on. The commented line at the end turns it off if Step 4 turned it on.

In [ ]:
# APIM = (f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/rg-foundry-core-{SUFFIX}'
#         f'/providers/Microsoft.ApiManagement/service/apim-foundry-{SUFFIX}')
# CONNECTION_API = os.environ.get('NEWS_CONNECTION_API_VERSION', ident.CONNECTION_API_VERSIONS[0])
#
#
# def purge(path):
#     """Delete a user or app, then purge it from the recycle bin."""
#     object_id = graph('GET', f'{path}?$select=id')['id']
#     graph('DELETE', path)
#     graph('DELETE', f'/directory/deletedItems/{object_id}')
#
#
# DELETE = {
#     'agent': lambda name: call('DELETE', f'{PROJECT_ENDPOINT}/agents/{name}?api-version=v1&force=true', FOUNDRY),
#     'connection': lambda name: call('DELETE', ident.connection_url(SUBSCRIPTION_ID, SUFFIX, name, CONNECTION_API), ARM),
#     'apim-api': lambda name: call('DELETE', f'{APIM}/apis/{name}?api-version=2025-09-01-preview&deleteRevisions=true', ARM),
#     'apim-named-value': lambda name: call('DELETE', f'{APIM}/namedValues/{name}?api-version=2025-09-01-preview', ARM),
#     'resource-group': lambda name: subprocess.run(f'az group delete -n {name} --yes', shell=True, capture_output=True, text=True),
#     'user': lambda object_id: purge(f'/users/{object_id}'),
#     'app': lambda app_id: purge(f"/applications(appId='{app_id}')"),
# }
# for description, kind, target in ident.teardown_inventory(dict(os.environ), SUFFIX):
#     try:
#         result = DELETE[kind](target)
#         status = getattr(result, 'status_code', getattr(result, 'returncode', 'done'))
#     except RuntimeError as e:
#         status = str(e)[:120]
#     print(f'  {description:<90} {status}')
#
# lines = ENV_FILE.read_text().splitlines()
# written = WRITTEN + ['NEWS_MCP_WRAPPER_URL', 'NEWS_SCIENCE_REVOKED_AT']
# ENV_FILE.write_text('\n'.join(l for l in lines if l.split('=', 1)[0].strip() not in written) + '\n')
# print(f'Removed {", ".join(written)} from .env')
#
# # Optional: turn the Temporary Access Pass method off again (only if Step 4 turned it on).
# # graph('PATCH', TAP_POLICY, {'@odata.type': ident.tap_policy_patch()['@odata.type'], 'state': 'disabled'}, token=admin_graph())

To check that nothing is left, run these in a terminal (`{suffix}` and the ids are in the cells above):

```bash
az group exists -n rg-foundry-news-mcp-{suffix}                                   # false
az ad app list --display-name news-mcp-{suffix} --query "length(@)"               # 0
az ad app list --display-name news-desk-client-{suffix} --query "length(@)"       # 0
az ad user show --id news-sports-desk@<your-domain>                               # not found
az rest --method GET --url "https://management.azure.com/subscriptions/<sub>/resourceGroups/rg-foundry-core-{suffix}/providers/Microsoft.ApiManagement/service/apim-foundry-{suffix}/apis/news-mcp?api-version=2025-09-01-preview"   # 404
```